In [1]:
!pip install imbalanced-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE # The secret weapon for imbalance

# 1. LOAD DATA
# We assume standard file names from the Kaggle dataset. 
# If yours are different, rename them here.
print("Loading Data...")
app = pd.read_csv('application_record.csv')
credit = pd.read_csv('credit_record.csv')

# 2. DEFINE "GOOD" VS "BAD" (The Target Variable)
# The prompt asks: "How do you define if a customer is good or bad?"
# Logic: If a user is late by > 60 days (Status 2, 3, 4, 5), they are "Bad" (1). 
# Otherwise, they are "Good" (0).

print("Constructing Target Variable...")
# Convert status to numeric, force errors to NaN
credit['STATUS'] = pd.to_numeric(credit['STATUS'], errors='coerce').fillna(0)

# Find IDs where the user was ever late > 60 days (Status >= 2)
bad_ids = credit[credit['STATUS'] >= 2]['ID'].unique()

# Create the label column in the main app dataframe
# 1 = Bad (High Risk), 0 = Good (Low Risk)
app['Target'] = app['ID'].apply(lambda x: 1 if x in bad_ids else 0)

print(f"Class Balance Before Fix:\n{app['Target'].value_counts()}")
# You will likely see HUGE imbalance here (e.g., 98% Good, 2% Bad)

# 3. PRE-PROCESSING (Cleaning)
# Drop columns with too many missing values or irrelevant IDs
app = app.drop(['ID', 'OCCUPATION_TYPE'], axis=1) 

# Encode text columns (Gender, Car, etc.) into numbers
# Machine Learning models only understand numbers, not "Male"/"Female"
le = LabelEncoder()
for col in app.columns:
    if app[col].dtype == 'object':
        app[col] = le.fit_transform(app[col].astype(str))

# Fill missing numerical values with the median
app.fillna(app.median(), inplace=True)

# 4. SPLIT DATA
X = app.drop('Target', axis=1) # The Features (Age, Income, etc.)
y = app['Target']              # The Answer (Good/Bad)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. FIX IMBALANCE (SMOTE)
# The prompt asks: "How can you overcome this problem?"
# We use SMOTE to synthesize new "Bad" examples so the model can learn.
print("\nApplying SMOTE to fix imbalance...")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Class Balance After SMOTE:\n{y_train_balanced.value_counts()}")
# Now it should be 50/50!

# 6. TRAIN MODEL (Random Forest)
print("\nTraining Random Forest Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_balanced, y_train_balanced)

# 7. EVALUATE
print("\n--- MODEL RESULTS ---")
predictions = model.predict(X_test)

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

# Classification Report (Precision, Recall, F1)
print("\nReport:")
print(classification_report(y_test, predictions))

# Feature Importance (What matters most?)
importance = pd.Series(model.feature_importances_, index=X.columns)
print("\nTop 5 Factors for Credit Approval:")
print(importance.nlargest(5))


Matplotlib is building the font cache; this may take a moment.


Loading Data...
Constructing Target Variable...
Class Balance Before Fix:
Target
0    437941
1       616
Name: count, dtype: int64

Applying SMOTE to fix imbalance...
Class Balance After SMOTE:
Target
0    350339
1    350339
Name: count, dtype: int64

Training Random Forest Model...

--- MODEL RESULTS ---

Confusion Matrix:
[[87489   113]
 [   76    34]]

Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     87602
           1       0.23      0.31      0.26       110

    accuracy                           1.00     87712
   macro avg       0.62      0.65      0.63     87712
weighted avg       1.00      1.00      1.00     87712


Top 5 Factors for Credit Approval:
CNT_FAM_MEMBERS     0.159154
AMT_INCOME_TOTAL    0.156686
DAYS_BIRTH          0.150879
DAYS_EMPLOYED       0.108095
FLAG_OWN_REALTY     0.077161
dtype: float64
